# OSL PPO — GRU


In [ ]:
import os, sys, subprocess

if os.path.isdir('/content'):
    REPO_URL = 'https://github.com/InHyunseo/Brain-inspired-OSL.git'
    REPO_DIR = '/content/2d-osl'
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
else:
    REPO_DIR = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(REPO_DIR, 'src')) and REPO_DIR != os.path.dirname(REPO_DIR):
        REPO_DIR = os.path.dirname(REPO_DIR)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('repo:', REPO_DIR, '\ncwd :', os.getcwd())


## Smoke check


In [ ]:
import torch, numpy as np
from src.envs.osl_env import OslEnv
from src.models.policy import Policy

env = OslEnv()
obs, info = env.reset(seed=0)
print('obs', obs.shape, 'action_space', env.action_space.shape)

policy = Policy(backbone='gru', gru_hidden=421, critic_type='mlp')
print('backbone:', policy.backbone.describe())
print('actor params :', sum(p.numel() for p in policy.actor_parameters()))
print('critic params:', sum(p.numel() for p in policy.critic_parameters()))

## Training


In [ ]:
# Env
ENV_KW = dict(
    sensor_spacing_mm=0.15,
    episode_seconds=120.0,
    arena_width_mm=80.0, arena_height_mm=120.0,
    source_x_mm=40.0, source_y_mm=100.0,
    gaussian_sigma_mm=30.0, success_radius_mm=7.5,
)

# Trainer (PPOConfig)
PPO_KW = dict(
    rollout_steps=128, num_envs=8, parallel_envs=True,
    update_epochs=4, minibatch_envs=4,
    gamma=0.99, gae_lambda=0.95, clip_epsilon=0.2,
    entropy_coef=0.005, value_loss_coef=0.5,
    actor_lr=3e-4, critic_lr=1e-3,
    actor_max_grad_norm=0.5, critic_max_grad_norm=0.5,
    log_std_init=-0.5,
    backbone='gru', gru_hidden=421,
    critic_type='mlp', critic_hidden=(64, 64),  # set critic_type='recurrent' for GRU critic
    eval_interval_updates=10, eval_episodes=3,
    log_every_updates=1, checkpoint_every_timesteps=100_000,
    seed=0,
    device='auto',  # 'cpu' to force CPU
)

In [ ]:
# ===== HYPERPARAMETERS — edit freely =====
# Curriculum: list of (noise_stage, noise_strength, env_steps).
# Bump-field noise model (see src/envs/odor_field.py):
#   stage 0 = clean Gaussian, no perturbation.
#   stage 1 = STATIC bump field — many local Gaussian bumps frozen at reset.
#   stage 2 = DYNAMIC bump field — bumps drift + AR(1) amplitude oscillation
#            + occasional respawn. Hydrodynamic-feeling local turbulence.
# `strength` is the curriculum scalar α ∈ [0, 1] that linearly scales all
# bump parameters: amp_max, n_bumps, drift_speed, lifetime_inv, respawn_prob.
# Per-bump sigma is capped near the source so the global plume gradient is
# preserved even at α = 1.0.
PHASES = [
    # --- Stage 0 (clean): success-radius curriculum, spawn fixed at 55-70mm ---
    # Start with a loose goal radius (20mm) so the agent gets early successes,
    # then tighten toward the real 7.5mm target. 1M steps per radius.
    (0, 0.0, 1_500_000, 20.0),
    (0, 0.0, 500_000, 10.0),
    (0, 0.0, 500_000,  7.5),   # real target; extra steps to consolidate
    # --- Noise curriculum (goal radius fixed at 7.5mm); stop at 0.3 (beyond it fails) ---
]

# Resume: set RESUME_FROM to a checkpoint .pt to continue a stopped run; the
# curriculum loop auto-skips phases already covered by the restored step count.
# Leave '' to start a fresh run.
RESUME_FROM = ''  # e.g. 'runs/ppo_gru_nb_20260531_113633/checkpoints/ckpt_1500000.pt'
# ==========================================

import os, time, json, torch
from src.agents.ppo_agent import PPOConfig, PPOTrainer

if RESUME_FROM:
    RUN_DIR = os.path.dirname(os.path.dirname(RESUME_FROM))  # .../checkpoints/x.pt -> run dir
    print('[resume] run_dir', RUN_DIR)
else:
    RUN_DIR = os.path.join('runs', f'ppo_gru_nb_{time.strftime("%Y%m%d_%H%M%S")}')
    os.makedirs(os.path.join(RUN_DIR, 'plots'), exist_ok=True)
    with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
        json.dump({'env': ENV_KW, 'ppo': PPO_KW, 'phases': PHASES}, f, indent=2)
    print('[run_dir]', RUN_DIR)

env_cfg = {**ENV_KW, 'noise_stage': 0, 'noise_strength': 0.0, 'seed': PPO_KW['seed']}
cfg = PPOConfig(**PPO_KW)

trainer = PPOTrainer(env_cfg, cfg, run_dir=RUN_DIR)
if RESUME_FROM:
    trainer.load_checkpoint(RESUME_FROM)
summary = {}
try:
    for stage, strength, steps, goal_radius in PHASES:
        print(f'\n=== Phase noise_stage={stage} strength={strength} goal_radius={goal_radius} steps={steps} ===')
        trainer.set_noise_stage(stage, strength)
        trainer.runner.set_success_radius(goal_radius)
        trainer.env_config['success_radius_mm'] = goal_radius  # so eval uses the same goal
        summary = trainer.train(phase_timesteps=steps)
    trainer.save_final(summary)
    print('\n[final summary]\n' + json.dumps(summary, indent=2))
finally:
    trainer.close()


## Training curve


In [ ]:
# ===== Training curve =====
import json, os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as DisplayImage, display

plt.rcParams.update({
    "font.size": 10, "axes.titlesize": 12, "axes.labelsize": 10,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9,
})
FIGSIZE = (6, 4.5)
FIG_DIR = os.path.join(RUN_DIR, "analysis")
os.makedirs(FIG_DIR, exist_ok=True)

rows = [json.loads(l) for l in open(os.path.join(RUN_DIR, "training_log.jsonl"))]
ev = [(r["total_steps"], r["eval_success_rate"]) for r in rows if "eval_success_rate" in r]
ev_x = [s / 1e6 for s, _ in ev]
ev_y = [y for _, y in ev]

def smooth(y, w=151):
    y = np.asarray(y, dtype=float)
    if len(y) < 3:
        return y
    w = min(w, len(y) if len(y) % 2 == 1 else len(y) - 1)
    if w < 3:
        return y
    pad = w // 2
    return np.convolve(np.pad(y, pad, mode="edge"), np.ones(w) / w, mode="valid")

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(ev_x, smooth(ev_y), color="#1f77b4", lw=2.4)
ax.set_xlabel("Steps (M)")
ax.set_ylim(-0.02, 1.02)
ax.set_title("Success ratio")
ax.grid(alpha=0.3)
# phase boundaries (edit if your curriculum changes)
for b in (1.5, 2.0):
    ax.axvline(b, color="gray", ls="--", lw=1.2)
for cx, name in ((0.75, "phase 0"), (1.75, "phase 1"), (2.25, "phase 2")):
    ax.text(cx, 1.04, name, ha="center", va="bottom", fontsize=10, color="dimgray")
fig.tight_layout()
out = os.path.join(FIG_DIR, "training_curve.png")
fig.savefig(out, dpi=150)
plt.close(fig)
print("saved", out)
display(DisplayImage(data=open(out, "rb").read(), format="png"))


## Trajectory PNG


In [ ]:
# ===== Trajectory PNG =====
EVAL_NOISE_STAGE = 2
EVAL_NOISE_STRENGTH = 0.3      # bump-field alpha at eval time
EVAL_SEED_BASE = 20_000
EVAL_MAX_TRIALS = 500          # upper bound on seeds to scan
TRAJ_N_SEEDS = 10              # number of trajectories to overlay
TRAJ_MIN_ACTIVE = 1            # prefer episodes with at least this many active-sensing events
TRAJ_MAX_ACTIVE = 300          # filters out pure spinning / pathological episodes
TRAJ_SEEDS = None              # None = auto-pick; or set a list like [20000, 20001]
EVAL_CKPT_PATH = globals().get('EVAL_CKPT_PATH', '')
EVAL_RUN_DIR = globals().get('EVAL_RUN_DIR', globals().get('RUN_DIR', ''))

import os, glob, torch, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
try:
    from IPython.display import Image as DisplayImage, display
except ImportError:
    DisplayImage = None
    def display(_):
        pass

from src.envs.osl_env import EnvConfig, OslEnv
from src.agents.ppo_agent import PPOConfig
from src.models.policy import Policy
from src.utils.plotter import _plume_field

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def resolve_eval_ckpt():
    if EVAL_CKPT_PATH:
        p = Path(EVAL_CKPT_PATH)
        if p.is_file():
            return str(p)
        raise FileNotFoundError(f'EVAL_CKPT_PATH does not exist: {p}')
    if EVAL_RUN_DIR:
        p = Path(EVAL_RUN_DIR)
        if p.is_file():
            return str(p)
        candidates = sorted(list((p / 'checkpoints').glob('*.pt')) + list(p.glob('*.pt')))
        if candidates:
            return str(candidates[-1])
    candidates = sorted(
        glob.glob('runs/ppo_gru*/ckpt_final.pt')
        + glob.glob('runs/ppo_gru*/checkpoints/*.pt')
        + glob.glob('runs/ppo_gru*/*.pt')
    )
    if candidates:
        return candidates[-1]
    raise FileNotFoundError('Set EVAL_CKPT_PATH or EVAL_RUN_DIR to a PPO+GRU checkpoint/run directory.')

ckpt_path = resolve_eval_ckpt()
ckpt_file = Path(ckpt_path)
plot_dir = (ckpt_file.parent.parent if ckpt_file.parent.name == 'checkpoints' else ckpt_file.parent) / 'plots'
print('using checkpoint', ckpt_path)

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
EVAL_ENV_KW = dict(ckpt.get('env_config', ENV_KW))
print('checkpoint steps', ckpt.get('training_state', {}).get('total_steps'))
agent_config = dict(ckpt['agent_config'])
if 'critic_type' not in agent_config:
    has_recurrent_critic = any(k.startswith('critic.cell.') for k in ckpt['policy_state_dict'])
    agent_config['critic_type'] = 'recurrent' if has_recurrent_critic else 'mlp'
cfg = PPOConfig(**agent_config)
policy = Policy(
    weights_csv=cfg.weights_csv, metadata_csv=cfg.metadata_csv,
    latent_dim=cfg.latent_dim, message_passing_steps=cfg.message_passing_steps,
    log_std_init=cfg.log_std_init,
    backbone=cfg.backbone, gru_hidden=cfg.gru_hidden,
    critic_type=cfg.critic_type, critic_hidden=cfg.critic_hidden,
).to(device)
policy.load_state_dict(ckpt['policy_state_dict']); policy.eval()

def make_eval_env(seed):
    cfg_dict = {**EVAL_ENV_KW, 'noise_stage': EVAL_NOISE_STAGE,
                'noise_strength': EVAL_NOISE_STRENGTH, 'seed': seed}
    return OslEnv(EnvConfig.from_dict(cfg_dict))

def _run_episode(seed, collect_traj=False):
    """Roll out one deterministic episode; return summary (+ optional trajectory)."""
    env = make_eval_env(seed)
    obs, _ = env.reset(seed=seed)
    actor_state, critic_state = policy.initial_states(1, device)
    mask = torch.zeros(1, 1, device=device)
    ret, active_sensing, success = 0.0, 0, False
    traj_x, traj_y, active_x, active_y = [], [], [], []
    for t in range(env.max_steps):
        if collect_traj:
            traj_x.append(env.x_mm); traj_y.append(env.y_mm)
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            a, _, _, n_as, n_cs = policy.act(
                obs_t, actor_state, critic_state, mask, deterministic=True
            )
        obs, r, term, trunc, info = env.step(a.squeeze(0).cpu().numpy())
        ret += float(r)
        if info.get('event_is_high_cast_like'):
            active_sensing += 1
            if collect_traj:
                active_x.append(env.x_mm); active_y.append(env.y_mm)
        done = bool(term or trunc)
        if done:
            success = bool(info.get('success', False))
            mask.fill_(0.0)
        else:
            mask.fill_(1.0)
        actor_state = n_as * mask; critic_state = n_cs * mask
        if done:
            break
    return {
        'seed': seed, 'return': ret, 'success': success,
        'active_sensing': active_sensing, 'steps': t + 1,
        'traj_x': traj_x, 'traj_y': traj_y,
        'active_x': active_x, 'active_y': active_y,
        'env': env if collect_traj else None,
    }

def find_trajectory_seeds(n_to_find=TRAJ_N_SEEDS, min_active=TRAJ_MIN_ACTIVE,
                          max_active=TRAJ_MAX_ACTIVE, max_trials=EVAL_MAX_TRIALS,
                          seed_base=EVAL_SEED_BASE):
    selected, successful = [], []
    print(f'seed scan: prefer success + active_sensing in [{min_active}, {max_active}], '
          f'alpha={EVAL_NOISE_STRENGTH}, up to {max_trials} trials')
    for trial in range(max_trials):
        seed = seed_base + trial
        r = _run_episode(seed, collect_traj=False)
        if not r['success']:
            continue
        successful.append(r)
        if min_active <= r['active_sensing'] <= max_active:
            selected.append(r)
            print(f"  seed={seed} return={r['return']:.2f} active_sensing={r['active_sensing']}")
        if len(selected) >= n_to_find:
            break
    if len(selected) < n_to_find:
        chosen = {r['seed'] for r in selected}
        fill = [r for r in successful if r['seed'] not in chosen]
        selected.extend(fill[:max(0, n_to_find - len(selected))])
        print(f'filled with success-only seeds: {len(selected)}/{n_to_find}')
    if len(selected) < n_to_find:
        chosen = {r['seed'] for r in selected}
        fallback = [seed_base + i for i in range(max_trials) if seed_base + i not in chosen]
        selected.extend({'seed': s} for s in fallback[:max(0, n_to_find - len(selected))])
        print(f'warning: included fallback seeds without success filtering: {len(selected)}/{n_to_find}')
    return [int(r['seed']) for r in selected[:n_to_find]]

selected_seeds = list(TRAJ_SEEDS) if TRAJ_SEEDS is not None else find_trajectory_seeds()
print('plotting seeds', selected_seeds)
rollouts = [_run_episode(seed, collect_traj=True) for seed in selected_seeds]
print(f"success={sum(r['success'] for r in rollouts)}/{len(rollouts)} "
      f"mean_steps={np.mean([r['steps'] for r in rollouts]):.1f} "
      f"mean_active_sensing={np.mean([r['active_sensing'] for r in rollouts]):.1f}")

plot_dir.mkdir(parents=True, exist_ok=True)
png_path = plot_dir / (
    f'ppo_gru_trajectories_n{len(rollouts)}_stage{EVAL_NOISE_STAGE}_'
    f'alpha{EVAL_NOISE_STRENGTH}.png'
)

# Background field is taken from the first selected seed. In dynamic/noisy
# fields each seed has its own bumps; using one field keeps the overlay readable.
field, W, H = _plume_field(rollouts[0]['env'])
env_cfg = rollouts[0]['env'].cfg
fig, ax = plt.subplots(figsize=(5.2, 7.6))
fig.patch.set_facecolor('black')
ax.set_facecolor('black')
ax.imshow(field, extent=[0.0, W, 0.0, H], origin='lower', cmap='magma',
          vmin=0.0, vmax=max(1e-6, float(field.max()) * 1.2))
colors = plt.cm.tab10(np.linspace(0.0, 1.0, max(10, len(rollouts))))
for idx, result in enumerate(rollouts):
    color = colors[idx % len(colors)]
    ax.plot(result['traj_x'], result['traj_y'], color=color, linewidth=2.0, alpha=0.86)
    if result['active_x']:
        ax.scatter(result['active_x'], result['active_y'], color='white', marker='*', s=48,
                   edgecolors='black', linewidths=0.35, alpha=0.75, zorder=10)
    ax.scatter([result['traj_x'][0]], [result['traj_y'][0]], color='white', marker='o', s=42,
               edgecolors='black', linewidths=0.45, alpha=0.9, zorder=11)
    ax.scatter([result['traj_x'][-1]], [result['traj_y'][-1]], color=color, marker='X', s=68,
               edgecolors='black', linewidths=0.45, zorder=11)
ax.scatter([env_cfg.source_x_mm], [env_cfg.source_y_mm], color='lime', marker='P', s=150,
           edgecolors='black', linewidths=0.6, zorder=12)
ax.add_patch(plt.Circle((env_cfg.source_x_mm, env_cfg.source_y_mm), env_cfg.success_radius_mm,
                        color='white', fill=False, linewidth=1.2, alpha=0.85))
ax.set_xlim(0.0, W); ax.set_ylim(0.0, H)
ax.set_xlabel('x (mm)', color='white'); ax.set_ylabel('y (mm)', color='white')
ax.set_aspect('equal')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('white')
fig.tight_layout()
fig.savefig(png_path, dpi=200, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close(fig)
print('[PNG] Saved to', png_path)
if DisplayImage is not None:
    display(DisplayImage(filename=str(png_path)))


## Noise sweep


In [ ]:
# ===== Noise sweep =====
import os, glob, json, torch, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.envs.osl_env import EnvConfig, OslEnv
from src.agents.ppo_agent import PPOConfig
from src.models.policy import Policy

SWEEP = [
    (0, 0.0), (1, 0.3), (1, 0.6), (1, 1.0),
    (2, 0.3), (2, 0.6), (2, 1.0),
]
SWEEP_EPISODES = 60   # per condition, matching baseline.ipynb
SWEEP_SEED_BASE = 30_000
NOISE_SWEEP_CKPT_PATH = globals().get('EVAL_CKPT_PATH', '')
NOISE_SWEEP_RUN_DIR = globals().get('EVAL_RUN_DIR', globals().get('RUN_DIR', ''))
ACCENT = globals().get('ACCENT', 'steelblue')
SWEEP_ENV_KW = globals().get('ENV_KW', dict(
    sensor_spacing_mm=0.15,
    episode_seconds=120.0,
    arena_width_mm=80.0, arena_height_mm=120.0,
    source_x_mm=40.0, source_y_mm=100.0,
    gaussian_sigma_mm=30.0, success_radius_mm=7.5,
))

def resolve_noise_sweep_ckpt():
    if NOISE_SWEEP_CKPT_PATH:
        p = Path(NOISE_SWEEP_CKPT_PATH)
        if p.is_file():
            return str(p)
        raise FileNotFoundError(f'NOISE_SWEEP_CKPT_PATH does not exist: {p}')
    if NOISE_SWEEP_RUN_DIR:
        p = Path(NOISE_SWEEP_RUN_DIR)
        if p.is_file():
            return str(p)
        candidates = sorted(list((p / 'checkpoints').glob('*.pt')) + list(p.glob('*.pt')))
        if candidates:
            return str(candidates[-1])
    candidates = sorted(
        glob.glob('runs/ppo_gru*/ckpt_final.pt')
        + glob.glob('runs/ppo_gru*/checkpoints/*.pt')
        + glob.glob('runs/ppo_gru*/*.pt')
    )
    if candidates:
        return candidates[-1]
    raise FileNotFoundError('Set EVAL_CKPT_PATH or EVAL_RUN_DIR to a PPO+GRU checkpoint/run directory.')

CKPT_PATH = resolve_noise_sweep_ckpt()
_ckpt_file = Path(CKPT_PATH)
_out_dir = _ckpt_file.parent.parent if _ckpt_file.parent.name == 'checkpoints' else _ckpt_file.parent
os.makedirs(_out_dir, exist_ok=True)
print('using checkpoint', CKPT_PATH)

_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_ckpt = torch.load(CKPT_PATH, map_location=_device, weights_only=False)
SWEEP_ENV_KW = dict(_ckpt.get('env_config', SWEEP_ENV_KW))
print('checkpoint steps', _ckpt.get('training_state', {}).get('total_steps'))
print('eval base env', {k: SWEEP_ENV_KW.get(k) for k in ['success_radius_mm', 'episode_seconds', 'noise_stage', 'noise_strength']})
_agent_config = dict(_ckpt['agent_config'])
if 'critic_type' not in _agent_config:
    _has_recurrent_critic = any(k.startswith('critic.cell.') for k in _ckpt['policy_state_dict'])
    _agent_config['critic_type'] = 'recurrent' if _has_recurrent_critic else 'mlp'
_agent_config.setdefault('critic_hidden', (64, 64))
_cfg = PPOConfig(**_agent_config)
_policy = Policy(
    weights_csv=_cfg.weights_csv, metadata_csv=_cfg.metadata_csv,
    latent_dim=_cfg.latent_dim, message_passing_steps=_cfg.message_passing_steps,
    log_std_init=_cfg.log_std_init,
    backbone=_cfg.backbone, gru_hidden=_cfg.gru_hidden,
    critic_type=_cfg.critic_type, critic_hidden=_cfg.critic_hidden,
).to(_device)
_policy.load_state_dict(_ckpt['policy_state_dict'])
_policy.eval()

def _sweep_episode(stage, strength, seed):
    cfg_dict = {**SWEEP_ENV_KW, 'noise_stage': int(stage), 'noise_strength': float(strength), 'seed': int(seed)}
    env = OslEnv(EnvConfig.from_dict(cfg_dict))
    obs, _ = env.reset(seed=seed)
    actor_state, critic_state = _policy.initial_states(1, _device)
    mask = torch.zeros(1, 1, device=_device)
    ret, casts, success, steps = 0.0, 0, False, 0
    for t in range(env.max_steps):
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=_device).unsqueeze(0)
        with torch.no_grad():
            action, _, _, next_actor_state, next_critic_state = _policy.act(
                obs_t, actor_state, critic_state, mask, deterministic=True)
        obs, reward, term, trunc, info = env.step(action.squeeze(0).cpu().numpy())
        ret += float(reward)
        steps = t + 1
        if info.get('event_is_high_cast_like'):
            casts += 1
        done = bool(term or trunc)
        mask.fill_(0.0 if done else 1.0)
        actor_state = next_actor_state * mask
        critic_state = next_critic_state * mask
        if done:
            success = bool(info.get('success', False))
            break
    return {'success': success, 'steps': steps, 'return': ret, 'casts': casts}

def evaluate(stage, strength, n_episodes=SWEEP_EPISODES, seed_base=SWEEP_SEED_BASE):
    episodes = [_sweep_episode(stage, strength, seed_base + i) for i in range(n_episodes)]
    succ = np.asarray([int(ep['success']) for ep in episodes])
    steps = np.asarray([ep['steps'] for ep in episodes])
    rets = np.asarray([ep['return'] for ep in episodes], dtype=float)
    casts = np.asarray([ep['casts'] for ep in episodes])
    succ_steps = steps[succ == 1]
    cast_frac = casts / np.maximum(steps, 1)
    return {
        'stage': stage, 'strength': strength, 'n': n_episodes,
        'success_rate': float(succ.mean()),
        'mean_steps_all': float(steps.mean()),
        'mean_steps_success': float(succ_steps.mean()) if len(succ_steps) else float('nan'),
        'mean_return': float(rets.mean()),
        'mean_casts': float(casts.mean()),
        'cast_fraction': float(cast_frac.mean()),
    }

rows = [evaluate(stage, strength, n_episodes=SWEEP_EPISODES) for stage, strength in SWEEP]

print(f"{'stage':>5} {'α':>4} {'success':>8} {'steps(succ)':>12} {'return':>8} {'casts':>7} {'cast%':>7}")
for r in rows:
    print(f"{r['stage']:>5} {r['strength']:>4.1f} {r['success_rate']:>7.0%} "
          f"{r['mean_steps_success']:>12.0f} {r['mean_return']:>8.2f} "
          f"{r['mean_casts']:>7.1f} {r['cast_fraction']:>6.1%}")

labels = [f"s{r['stage']}·α{r['strength']}" for r in rows]
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].bar(labels, [r['success_rate'] for r in rows], color=ACCENT)
ax[0].set_ylim(0, 1); ax[0].set_ylabel('success rate')
ax[0].set_title('Success ratio')
ax[0].tick_params(axis='x', rotation=30)
ax[1].bar(labels, [r['mean_steps_success'] for r in rows], color=ACCENT)
ax[1].set_ylabel('steps to source')
ax[1].set_title('Steps to source')
ax[1].tick_params(axis='x', rotation=30)
ax[2].bar(labels, [100.0 * r['cast_fraction'] for r in rows], color=ACCENT)
ax[2].set_ylabel('cast steps (%)')
ax[2].set_title('Cast fraction')
ax[2].tick_params(axis='x', rotation=30)
fig.tight_layout()

fig.savefig(_out_dir / 'noise_sweep.png', dpi=150)
with open(_out_dir / 'noise_sweep.json', 'w') as f:
    json.dump(rows, f, indent=2)
plt.show()
print('\n[saved]', _out_dir / 'noise_sweep.{png,json}')


## Jacobian


In [ ]:
# ===== Jacobian =====
import os, glob, json, torch, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.envs.osl_env import EnvConfig, OslEnv
from analysis.osl2d.policy_adapter import Policy2DAdapter
from analysis.osl2d.jacobian import jacobian_at

try:
    from IPython.display import Image as DisplayImage, display
except ImportError:
    DisplayImage = None
    def display(_):
        pass

JAC_NOISE_STAGE = globals().get('EVAL_NOISE_STAGE', 2)
JAC_NOISE_STRENGTH = globals().get('EVAL_NOISE_STRENGTH', 0.3)
JAC_SEED_BASE = 40_000
JAC_MAX_ROLLOUTS = 40
JAC_MAX_STEPS = 1200
JAC_SAMPLES_PER_LABEL = 12     # Jacobian is expensive; raise for final-quality plots.
JAC_STOCHASTIC = False         # set True if deterministic rollouts produce too few active-sensing samples
JAC_CKPT_PATH = globals().get('EVAL_CKPT_PATH', '')
JAC_RUN_DIR = globals().get('EVAL_RUN_DIR', globals().get('RUN_DIR', ''))
JAC_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
JAC_ENV_KW = globals().get('ENV_KW', dict(
    sensor_spacing_mm=0.15,
    episode_seconds=120.0,
    arena_width_mm=80.0, arena_height_mm=120.0,
    source_x_mm=40.0, source_y_mm=100.0,
    gaussian_sigma_mm=30.0, success_radius_mm=7.5,
))

plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
})

BLUE, RED, GRAY = '#1f77b4', '#d62728', '0.68'

def resolve_jac_ckpt():
    if JAC_CKPT_PATH:
        p = Path(JAC_CKPT_PATH)
        if p.is_file():
            return str(p)
        raise FileNotFoundError(f'JAC_CKPT_PATH does not exist: {p}')
    if JAC_RUN_DIR:
        p = Path(JAC_RUN_DIR)
        if p.is_file():
            return str(p)
        candidates = sorted(list((p / 'checkpoints').glob('*.pt')) + list(p.glob('*.pt')))
        if candidates:
            return str(candidates[-1])
    candidates = sorted(
        glob.glob('runs/ppo_gru*/ckpt_final.pt')
        + glob.glob('runs/ppo_gru*/checkpoints/*.pt')
        + glob.glob('runs/ppo_gru*/*.pt')
    )
    if candidates:
        return candidates[-1]
    raise FileNotFoundError('Set EVAL_CKPT_PATH or EVAL_RUN_DIR to a PPO+GRU checkpoint/run directory.')

CKPT_PATH = resolve_jac_ckpt()
ckpt_file = Path(CKPT_PATH)
OUT_DIR = ckpt_file.parent.parent if ckpt_file.parent.name == 'checkpoints' else ckpt_file.parent
ANALYSIS_DIR = OUT_DIR / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print('using checkpoint', CKPT_PATH)
_jac_payload = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
JAC_ENV_KW = dict(_jac_payload.get('env_config', JAC_ENV_KW))
print('checkpoint steps', _jac_payload.get('training_state', {}).get('total_steps'))

adapter = Policy2DAdapter.from_checkpoint(CKPT_PATH, device=JAC_DEVICE)
print('backbone:', adapter.backbone_kind, 'state_size:', adapter.state_size, 'device:', adapter.device)

def make_jac_env(seed):
    cfg = {**JAC_ENV_KW, 'noise_stage': int(JAC_NOISE_STAGE),
           'noise_strength': float(JAC_NOISE_STRENGTH), 'seed': int(seed)}
    return OslEnv(EnvConfig.from_dict(cfg))

def collect_points():
    samples = {'NORMAL': [], 'ACTIVE_SENSING': []}
    fallback_normal = []
    for rollout_i in range(JAC_MAX_ROLLOUTS):
        seed = JAC_SEED_BASE + rollout_i
        env = make_jac_env(seed)
        obs, _ = env.reset(seed=seed)
        h = adapter.initial_state()
        gen = torch.Generator(device=adapter.device)
        gen.manual_seed(seed)

        for t in range(min(env.max_steps, JAC_MAX_STEPS)):
            obs_t = np.asarray(obs, dtype=np.float32).copy()
            h_t = np.asarray(h, dtype=np.float32).copy()

            if JAC_STOCHASTIC:
                action, h_next = adapter.step_stochastic(obs_t, h_t, generator=gen)
            else:
                action, h_next = adapter.step_patched(obs_t, h_t)

            obs, reward, terminated, truncated, info = env.step(action)
            is_active = bool(info.get('event_is_high_cast_like', False))
            is_run = bool(info.get('event_is_run', False))
            is_quiet_nonactive = not is_active and not bool(info.get('event_is_spin_like', False))

            if is_active and len(samples['ACTIVE_SENSING']) < JAC_SAMPLES_PER_LABEL:
                samples['ACTIVE_SENSING'].append((obs_t, h_t, seed, t))
            elif is_run and len(samples['NORMAL']) < JAC_SAMPLES_PER_LABEL:
                samples['NORMAL'].append((obs_t, h_t, seed, t))
            elif is_quiet_nonactive and len(fallback_normal) < JAC_SAMPLES_PER_LABEL:
                fallback_normal.append((obs_t, h_t, seed, t))

            h = h_next
            if all(len(v) >= JAC_SAMPLES_PER_LABEL for v in samples.values()):
                return samples
            if terminated or truncated:
                break

    if len(samples['NORMAL']) < JAC_SAMPLES_PER_LABEL:
        need = JAC_SAMPLES_PER_LABEL - len(samples['NORMAL'])
        samples['NORMAL'].extend(fallback_normal[:need])
    return samples

samples = collect_points()
print('sample counts:', {k: len(v) for k, v in samples.items()})
if min(len(v) for v in samples.values()) == 0:
    raise RuntimeError(
        'Need at least one NORMAL and one ACTIVE_SENSING sample. Try increasing '
        'JAC_MAX_ROLLOUTS, changing JAC_NOISE_STRENGTH, or setting JAC_STOCHASTIC=True.'
    )

def eigs_for(label):
    all_eigs, dominant, meta = [], [], []
    for obs_t, h_t, seed, t in samples[label]:
        J = jacobian_at(adapter, obs_t, h_t)
        w = np.linalg.eigvals(J)
        top = w[int(np.argmax(np.abs(w)))]
        all_eigs.append(w)
        dominant.append(top)
        meta.append({'seed': int(seed), 'step': int(t)})
    return np.concatenate(all_eigs), np.asarray(dominant), meta

pack = {label: eigs_for(label) for label in ('NORMAL', 'ACTIVE_SENSING')}
summary = {}
for label, (E, D, meta) in pack.items():
    summary[label] = {
        'n_samples': int(len(D)),
        'n_eigenvalues': int(len(E)),
        'dominant_abs_mean': float(np.mean(np.abs(D))),
        'dominant_abs_std': float(np.std(np.abs(D))),
        'dominant_abs_imag_mean': float(np.mean(np.abs(D.imag))),
        'dominant_abs_imag_std': float(np.std(np.abs(D.imag))),
        'dominant_oscillatory_fraction': float(np.mean(np.abs(D.imag) > 1e-6)),
        'samples': meta,
    }

fig, ax = plt.subplots(2, 2, figsize=(10, 8))
theta = np.linspace(0, 2 * np.pi, 512)
for j, (label, title, color) in enumerate([
    ('NORMAL', 'Normal RUN', BLUE),
    ('ACTIVE_SENSING', 'Active sensing', RED),
]):
    E, D, _ = pack[label]
    a = ax[0, j]
    a.scatter(E.real, E.imag, s=4, alpha=0.18, color=GRAY, label='all modes')
    a.scatter(D.real, D.imag, s=34, alpha=0.95, color=color, edgecolors='black', linewidths=0.35,
              label='dominant |lambda|')
    a.plot(np.cos(theta), np.sin(theta), color='black', lw=0.7, alpha=0.45)
    a.axhline(0, color='black', lw=0.35, alpha=0.45)
    a.axvline(0, color='black', lw=0.35, alpha=0.45)
    lim = max(1.05, float(np.nanpercentile(np.abs(E), 99)) * 1.08)
    a.set_xlim(-lim, lim); a.set_ylim(-lim, lim)
    a.set_aspect('equal', adjustable='box')
    a.set_title(title)
    a.set_xlabel('Re(lambda)')
    a.set_ylabel('Im(lambda)')
    a.legend(loc='upper left', framealpha=0.9)

labels = ['NORMAL', 'ACTIVE_SENSING']
pretty = ['Normal', 'Active']
colors = [BLUE, RED]
metrics = [
    ('Dominant magnitude', [np.abs(pack[k][1]) for k in labels], '|lambda*|'),
    ('Dominant imaginary part', [np.abs(pack[k][1].imag) for k in labels], '|Im(lambda*)|'),
]
for a, (title, vals, ylabel) in zip(ax[1], metrics):
    means = [float(np.mean(v)) for v in vals]
    stds = [float(np.std(v)) for v in vals]
    x = np.arange(len(vals))
    a.bar(x, means, yerr=stds, color=colors, alpha=0.82, capsize=4)
    for xi, v in zip(x, vals):
        jitter = np.linspace(-0.08, 0.08, len(v)) if len(v) > 1 else np.array([0.0])
        a.scatter(np.full(len(v), xi) + jitter, v, color='black', s=18, alpha=0.65, zorder=4)
    a.set_xticks(x, pretty)
    a.set_ylabel(ylabel)
    a.set_title(title)
    a.grid(axis='y', alpha=0.25)

fig.suptitle(f'GRU hidden Jacobian: stage={JAC_NOISE_STAGE}, alpha={JAC_NOISE_STRENGTH}, n={JAC_SAMPLES_PER_LABEL}/label', y=0.995)
fig.tight_layout()
out_png = ANALYSIS_DIR / 'jacobian_active_vs_normal.png'
out_json = ANALYSIS_DIR / 'jacobian_active_vs_normal.json'
fig.savefig(out_png, dpi=180)
with open(out_json, 'w') as f:
    json.dump(summary, f, indent=2)
plt.show()
print('summary:', json.dumps({k: {kk: round(vv, 4) if isinstance(vv, float) else vv
                                  for kk, vv in val.items() if kk != 'samples'}
                              for k, val in summary.items()}, indent=2))
print('[saved]', out_png)
print('[saved]', out_json)
if DisplayImage is not None:
    display(DisplayImage(data=open(out_png, 'rb').read(), format='png'))
